# UC-CAT-3 — Schema Evolution: Add a Column

**Persona:** Data Engineer

**Goal:** Safely add a `cloud_cover_pct` column to an existing PostgreSQL-backed collection
using the schema evolution API:
1. Create a test collection with PostgreSQL driver configured
2. Inspect existing backups
3. Dry-run the evolution to inspect the diff without mutating state
4. Apply the evolution
5. Verify the new field appears in the OGC queryables endpoint
6. Cleanup the test collection

**Prerequisites:**
- A catalog already exists (pass `CATALOG_ID` or it defaults to `demo-catalog`)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- `driver:postgresql` installed on the platform

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`, `CATALOG_ID` (optional)

In [ ]:
import os
import json

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "ndvi_mosaic_2024")

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=60.0)
print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}  collection={COLLECTION_ID}")

In [ ]:
## Setup — Create test collection with PostgreSQL driver

Before schema evolution, we need a collection with PostgreSQL storage configured.
This cell creates it automatically so the notebook is self-contained.

import uuid

COLLECTION_ID = f"schema-evolution-test-{uuid.uuid4().hex[:8]}"

# Create the collection
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "Schema Evolution Test Collection",
    "description": "Test collection for schema evolution demo.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}

r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
assert r.status_code == 201, f"Failed to create collection: {r.status_code}: {r.text}"
print(f"Created test collection: {COLLECTION_ID}")

# Configure PostgreSQL driver
driver_config = {
    "enabled": True,
    "partition_type": "LIST",
    "sidecars": ["item_metadata", "stac_metadata"],
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionPostgresqlDriverConfig",
    content=json.dumps(driver_config),
)
assert r.status_code == 200, f"Failed to configure driver: {r.status_code}: {r.text}"

# Configure routing
routing_config = {
    "enabled": True,
    "operations": {
        "WRITE": [{"driver_id": "postgresql", "on_failure": "fatal"}],
        "READ": [{"driver_id": "postgresql", "on_failure": "fatal"}],
    },
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
    content=json.dumps(routing_config),
)
assert r.status_code == 200, f"Failed to configure routing: {r.status_code}: {r.text}"
print(f"Collection ready for schema evolution")

## Step 0 — Guard: confirm no reindex is running

Applying a schema evolution while an active reindex is in progress can produce inconsistent
index mappings. Always check before evolving.

In [ ]:
# Query the catalog task log for active REINDEX events
r = client.get(
    f"/logs/catalogs/{CATALOG_ID}/logs",
    params={"event_type": "REINDEX", "level": "INFO", "limit": 10},
)
print(r.status_code, r.text[:500])
# 200 expected; 404 is acceptable if no log backend is configured
assert r.status_code in (200, 404), (
    f"Unexpected status {r.status_code}: {r.text}"
)

if r.status_code == 200:
    log_entries = r.json()
    active_reindex = [
        e for e in (log_entries if isinstance(log_entries, list) else log_entries.get("entries", []))
        if e.get("status") in ("running", "accepted", "RUNNING", "ACCEPTED")
    ]
    assert len(active_reindex) == 0, (
        f"Active reindex detected — do not evolve schema while reindex is running: {active_reindex}"
    )
    print(f"No active reindex found ({len(log_entries if isinstance(log_entries, list) else log_entries.get('entries', []))} recent log entries checked). Safe to proceed.")
else:
    print("Log backend not available (404). Proceeding without reindex guard — verify manually.")

## Step 1 — Check existing backups

The evolution API creates a backup snapshot before applying changes. Listing existing backups
lets you track the history of schema changes and identify a restore point if needed.

In [ ]:
r = client.get(f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/backups")
print(r.status_code, r.text[:600])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

backups = r.json()
backup_list = backups if isinstance(backups, list) else backups.get("backups", [])
print(f"Existing backups: {len(backup_list)}")
for b in backup_list:
    print(f"  - {b.get('name', b.get('id', '?'))}  created={b.get('created_at', '?')}")

## Step 2 — Dry-run evolution (inspect diff)

Post an `EvolutionRequest` with `apply_safe=false` (or the platform-equivalent dry-run flag).
The server returns the diff of what *would* be applied without mutating the schema.
Always review the diff before applying.

In [ ]:
evolution_request = {
    "fields": [
        {
            "name": "cloud_cover_pct",
            "type": "numeric",
            "nullable": True,
            "description": "Cloud cover percentage [0–100] derived from MODIS QA band.",
        }
    ],
    "apply_safe": False,  # dry-run: compute diff, do not mutate
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(evolution_request),
)
print(r.status_code, r.text[:800])
assert r.status_code == 200, f"Expected 200 (dry-run), got {r.status_code}: {r.text}"

dry_run_result = r.json()
print("\n--- Dry-run diff ---")
print(json.dumps(dry_run_result, indent=2))

## Step 3 — Apply evolution

Re-post with `apply_safe=true`. The platform will:
1. Create a schema backup
2. Execute the `ALTER TABLE ... ADD COLUMN` DDL inside a transaction
3. Return a summary of applied changes

Only backwards-compatible operations (ADD COLUMN, CREATE INDEX) are allowed when `apply_safe=true`.
Destructive operations (DROP COLUMN, ALTER TYPE) are blocked.

In [ ]:
evolution_request_apply = {
    **evolution_request,
    "apply_safe": True,  # apply: mutate schema
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(evolution_request_apply),
)
print(r.status_code, r.text[:800])
assert r.status_code == 200, f"Expected 200 (apply), got {r.status_code}: {r.text}"

apply_result = r.json()
print("\n--- Applied changes ---")
print(json.dumps(apply_result, indent=2))

applied = apply_result if isinstance(apply_result, list) else apply_result.get("applied", [])
applied_names = [
    c.get("name", c.get("column", "")) for c in applied
] if applied else []
# Some platforms return the result nested differently; check broadly
result_str = json.dumps(apply_result)
assert "cloud_cover_pct" in result_str, (
    f"'cloud_cover_pct' not found in apply response: {result_str[:400]}"
)
print("'cloud_cover_pct' confirmed in applied changes.")

## Step 4 — Verify via queryables

The OGC API Queryables endpoint reflects the current physical schema. After evolution,
`cloud_cover_pct` must appear as a queryable property clients can use in CQL2 filter expressions.

In [ ]:
r = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables"
)
print(r.status_code, r.text[:800])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

queryables = r.json()
# Queryables follows JSON Schema: properties are under "properties" key
properties = queryables.get("properties", {})
queryable_names = list(properties.keys())
print(f"Queryable properties ({len(queryable_names)}): {queryable_names}")

assert "cloud_cover_pct" in queryable_names, (
    f"'cloud_cover_pct' not found in queryables. Available: {queryable_names}"
)
print("'cloud_cover_pct' is now a queryable field — schema evolution confirmed.")

## Edge Cases

### Destructive operations are blocked by `apply_safe=true`

The platform rejects DROP COLUMN when `apply_safe=true`. The cell below documents this
constraint without executing a destructive operation — you would need to temporarily
set `apply_safe=false` (which bypasses the guard) to perform irreversible DDL.

In [ ]:
# Document: DROP COLUMN with apply_safe=True must be rejected
drop_request = {
    "fields": [
        {
            "name": "cloud_cover_pct",
            "action": "drop",  # destructive operation
        }
    ],
    "apply_safe": True,
}

r = client.post(
    f"/admin/schemas/{CATALOG_ID}/{COLLECTION_ID}/evolve",
    content=json.dumps(drop_request),
)
print(f"DROP COLUMN with apply_safe=True: status={r.status_code}")
print(r.text[:400])

# The platform must reject destructive operations under apply_safe mode
# Expected: 422 Unprocessable Entity or 400 Bad Request
assert r.status_code in (400, 409, 422), (
    f"Expected 4xx rejection for DROP COLUMN under apply_safe=True, got {r.status_code}. "
    "If the platform accepted this, review DDL safety gates."
)
print(f"Confirmed: destructive evolution correctly rejected with {r.status_code}.")

## Notes on reindex ordering

- **Always verify** that no reindex is running before applying schema evolution (Step 0 above).
- After evolution, if the collection is indexed in Elasticsearch, the ES mapping must be
  updated separately — schema evolution only alters the PostgreSQL table. Trigger a
  reindex via `POST /search/catalogs/{CATALOG_ID}/reindex` once the column is populated.
- Column additions are `NOT NULL` safe only if a default value is provided or the column
  is nullable. The example above uses `nullable: true` to avoid a full table rewrite.
- There is no teardown cell for this notebook because the column addition is permanent.
  If you need to remove the column, use the admin SQL console with `apply_safe=false`
  and expect a full reindex to follow.

**To clean up the collection entirely**, run the teardown cell in
`catalog/02_create_collection_with_layerconfig.ipynb`.

In [ ]:
client.close()
print("Session closed.")

In [ ]:
## Teardown

Clean up the test collection created in the setup cell above.

r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
assert r.status_code == 204, f"Failed to delete collection: {r.status_code}: {r.text}"
print(f"Test collection {COLLECTION_ID} deleted.")